# 03 · Data Enrichment

Feature films released from 1980 to 2025 are enriched with human-authored open data:

- **Selection (Spark):** the 10 most popular films (IMDb votes) of every genre and year, and the first 3 principals (billing order) of each film.
- **Gathering (outside Spark):** [Wikidata](https://www.wikidata.org/) links each IMDb identifier (property [P345](https://www.wikidata.org/wiki/Property:P345)) to its English Wikipedia article and to the film's original language ([P364](https://www.wikidata.org/wiki/Property:P364)). The [Wikipedia API](https://en.wikipedia.org/w/api.php) returns the article text written by Wikipedia editors. No text is generated.
- **Cleaning and joins (Spark):** plot sections and biographies are extracted from the raw text, the schema is enforced, and every record keeps its source URL, a revision permalink and a note when information is missing.

Requests are serial, rate-limited and identified with a contact address, and every answer is cached, so an interrupted run resumes without asking again.

In [ ]:
import os
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
ROOT = next((path for path in (cwd, *cwd.parents) if (path / "src/imdb_loader.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Run the notebook from the IMDb repository or its notebooks folder.")
sys.path.insert(0, str(ROOT))

import csv
import json
import time

from pyspark import StorageLevel
from pyspark.sql import functions as F
from src import enrichment as enrich
from src import wiki_sources as wiki
from src.imdb_loader import data_root, prepared_snapshot, read_prepared, spark_session

DATA = data_root(ROOT)
snapshot = prepared_snapshot(DATA)
spark = spark_session("IMDb Data Enrichment")
tables = read_prepared(spark, snapshot)

PARAMETERS = dict(first_year=1980, last_year=2025, min_minutes=40, top_n=10, principals_per_film=3)
# Number of identifiers to gather per step: a small number for a first test, None for everything.
GATHER_LIMIT = 30
CONTACT = os.getenv("WIKI_CONTACT", "")  # e-mail sent in the User-Agent, required by Wikimedia

WORK = DATA / "enrichment"
IDS, RAW = WORK / "ids", WORK / "raw"
OUTPUT = WORK / "output" / snapshot.name
RAW.mkdir(parents=True, exist_ok=True)
print(f"Prepared snapshot: {snapshot.name}")

## Step 1 · Select films and principals (Spark)

A feature film is a `movie` lasting at least 40 minutes, as in exploration Q1. Popularity is the number of IMDb votes, as in Q2. Adult titles are excluded. A film appears once per genre it belongs to, so the number of distinct films is lower than genres × years × 10.

In [ ]:
selected = enrich.select_top_films(
    tables["title.basics"], tables["title.ratings"],
    first_year=PARAMETERS["first_year"], last_year=PARAMETERS["last_year"],
    min_minutes=PARAMETERS["min_minutes"], top_n=PARAMETERS["top_n"],
).persist(StorageLevel.MEMORY_AND_DISK)
chosen = enrich.select_principals(
    tables["title.principals"], selected, per_film=PARAMETERS["principals_per_film"]
).persist(StorageLevel.MEMORY_AND_DISK)

enrich.write_ids(selected.withColumnRenamed("tconst", "imdb_id"), "imdb_id", (IDS / "films").as_posix())
enrich.write_ids(chosen.withColumnRenamed("nconst", "imdb_id"), "imdb_id", (IDS / "people").as_posix())

print(f"Film-genre-year rows : {selected.count():,}")
print(f"Distinct films       : {selected.select('tconst').distinct().count():,}")
print(f"Film principals      : {chosen.count():,}")
print(f"Distinct people      : {chosen.select('nconst').distinct().count():,}")
selected.filter(F.col("startYear") == 2010).orderBy("genre", "genre_rank").show(12, truncate=40)

## Step 2 · Gather Wikidata links and Wikipedia text (outside Spark)

Start with `GATHER_LIMIT = 30` to check that everything works, then set it to `None` and run these cells again: cached answers are skipped. One request per film is needed (the API returns full articles one at a time); biographies are fetched 20 per request.

In [ ]:
client = wiki.PoliteClient(CONTACT, min_interval=0.5)
ids = (*wiki.read_ids(IDS / "films", "imdb_id"), *wiki.read_ids(IDS / "people", "imdb_id"))
start = time.monotonic()
added = wiki.fetch_wikidata(client, ids, RAW / "wikidata.jsonl", limit=GATHER_LIMIT)
print(f"Wikidata: {added:,} identifiers added in {time.monotonic() - start:.0f}s")

In [ ]:
start = time.monotonic()
films_added = wiki.fetch_articles(
    client, wiki.article_urls(RAW / "wikidata.jsonl", "tt"), RAW / "wikipedia_films.jsonl",
    intro_only=False, limit=GATHER_LIMIT,
)
people_added = wiki.fetch_articles(
    client, wiki.article_urls(RAW / "wikidata.jsonl", "nm"), RAW / "wikipedia_people.jsonl",
    intro_only=True, limit=GATHER_LIMIT,
)
print(f"Wikipedia: {films_added:,} film and {people_added:,} people articles added "
      f"in {time.monotonic() - start:.0f}s ({client.requests_sent:,} requests this session)")

## Step 3 · Clean, join and enforce the schema (Spark)

The raw caches are read with explicit schemas. The plot section is the level-2 section titled *Plot*, *Plot summary*, *Synopsis*, *Premise*, *Summary*, *Story* or *Storyline*; the biography is the article's introduction. Each record carries `source_url`, `source_revision_url` (the exact revision used) and `wikidata_uri`. When information is missing, `enrichment_status` is `not_available` and `enrichment_note` says why.

In [ ]:
links = enrich.wikidata_links(
    enrich.read_jsonl(spark, (RAW / "wikidata.jsonl").as_posix(), enrich.WIKIDATA_SCHEMA)
)
film_pages = enrich.article_sections(
    enrich.read_jsonl(spark, (RAW / "wikipedia_films.jsonl").as_posix(), enrich.ARTICLE_SCHEMA)
)
people_pages = enrich.article_sections(
    enrich.read_jsonl(spark, (RAW / "wikipedia_people.jsonl").as_posix(), enrich.ARTICLE_SCHEMA)
)

films = enrich.enrich_films(selected, links, film_pages)
principals = enrich.enrich_principals(chosen, tables["name.basics"], links, people_pages)
films.write.mode("overwrite").parquet((OUTPUT / "films").as_posix())
principals.write.mode("overwrite").parquet((OUTPUT / "principals").as_posix())

films = spark.read.parquet((OUTPUT / "films").as_posix())
principals = spark.read.parquet((OUTPUT / "principals").as_posix())
assert films.filter(F.col("source_url").isNull()).count() == 0, "Film without lineage"
assert principals.filter(F.col("source_url").isNull()).count() == 0, "Principal without lineage"
assert films.filter((F.col("enrichment_status") == "enriched") & F.col("synopsis").isNull()).count() == 0
print(f"Saved to {OUTPUT}")

## Results · Coverage and examples

In [ ]:
print("Films")
enrich.coverage(films).show(truncate=80)
print("Principals")
enrich.coverage(principals).show(truncate=80)

films.filter(F.col("enrichment_status") == "enriched").select(
    "tconst", "primaryTitle", "startYear", F.substring("synopsis", 1, 120).alias("synopsis"),
    "source_revision_url",
).orderBy(F.desc("numVotes")).show(5, truncate=60)
principals.filter(F.col("enrichment_status") == "enriched").select(
    "tconst", "primaryName", "category", F.substring("biography", 1, 100).alias("biography"), "source_url",
).show(5, truncate=60)

## Bonus · Original languages for exploration Q6

Wikidata's *original language of film* (P364) is exported in the format expected by `02_data_exploration.ipynb` (`data/reference/original_languages.csv`, one row per film and language with its source). Q6 can then run in `verified` mode on the enriched films.

In [ ]:
reference = DATA / "reference" / "original_languages.csv"
reference.parent.mkdir(parents=True, exist_ok=True)
languages = films.select(
    "tconst", F.explode("original_languages").alias("language"), F.col("wikidata_uri").alias("source_uri")
).filter(F.col("source_uri").isNotNull())
with reference.open("w", newline="", encoding="utf-8") as stream:
    writer = csv.writer(stream)
    writer.writerow(["tconst", "language", "source_uri"])
    rows = 0
    for row in languages.toLocalIterator():
        writer.writerow([row.tconst, row.language, row.source_uri])
        rows += 1
print(f"{rows:,} film-language rows written to {reference}")

In [ ]:
spark.stop()